# 01 · Extração de features (Paderborn / KAt-DataCenter)

**Objetivo:** ler os arquivos `.mat` do dataset (20 rolamentos × 4 condições × 20 medições = 1600 arquivos) e gerar **uma tabela de features**, com uma linha por medição (ou por segmento) e uma coluna por feature.

**Decisões de projeto:**

| Decisão | Por quê |
|---|---|
| Cada canal é lido diretamente de `Y.Data`, sem juntar canais | os canais têm taxas de amostragem diferentes (64 kHz, 4 kHz e 1 Hz). Combiná-los exigiria interpolação, que cria amostras artificiais |
| Taxa de amostragem fixa em `FS = 64 000` Hz | é a taxa nominal das correntes e da vibração no dataset. Uma taxa errada desloca todas as frequências calculadas |
| Features calculadas dentro do loop de leitura, com o sinal bruto descartado em seguida | só os números extraídos ficam na memória, e não os sinais completos |
| Processamento em paralelo com `joblib` (um processo por núcleo) | o tempo total é dividido pelo número de núcleos do processador |
| Features no tempo, na frequência, por Wavelet Packet de nível 3, no espectro de envelope (BPFO e BPFI) e em bandas laterais na corrente | combina os indicadores estatísticos usados no artigo com features ligadas à física do defeito |

Toda a lógica fica em `rolamentos.py`, na mesma pasta deste notebook. O notebook só configura, chama as funções e confere os resultados.

## 1. Configuração
- `PASTA_DADOS`: pasta com uma subpasta por rolamento (`dados/K001/…mat`).
- `N_SEGMENTOS`: em quantos pedaços cada sinal de 4 s é cortado. `1` replica o artigo (uma amostra por medição). Com `4`, saem 4 amostras de 1 s por medição. Os segmentos herdam o rolamento, então a divisão por grupo continua válida.
- `N_JOBS = -1`: usa todos os núcleos.

`TEM_PYWT` mostra se o PyWavelets está instalado. Sem ele, as energias `wpd_e*` são calculadas por bandas de FFT equivalentes.

In [12]:
# import time
from pathlib import Path

import numpy as np
import pandas as pd

import rolamentos as R

PASTA_DADOS = Path("dados")
ARQUIVO_SAIDA = Path("features.csv")
N_SEGMENTOS = 1
N_JOBS = -1

print("PyWavelets disponível:", R.TEM_PYWT)

PyWavelets disponível: True


## 2. Inventário dos arquivos
Antes de processar, confirmamos que os arquivos esperados estão lá. Cada rolamento deve ter **4 condições × 20 medições = 80 arquivos**. A tabela cruzada mostra na hora se falta alguma pasta ou condição.

O nome do arquivo é interpretado por expressão regular (`N15_M07_F10_KA04_1`). Daí saem condição, rolamento, número da medição, rotação, torque e força.

In [2]:
arquivos = R.listar_arquivos(PASTA_DADOS)
inventario = pd.DataFrame([R.interpretar_nome(a) for a in arquivos])
print(f"{len(arquivos)} arquivos, {inventario['rolamento'].nunique()} rolamentos")
pd.crosstab(inventario["rolamento"], inventario["condicao"], margins=True)

1600 arquivos, 20 rolamentos


condicao,N09_M07_F10,N15_M01_F10,N15_M07_F04,N15_M07_F10,All
rolamento,,,,,
K001,20,20,20,20,80
K002,20,20,20,20,80
K003,20,20,20,20,80
K004,20,20,20,20,80
K005,20,20,20,20,80
K006,20,20,20,20,80
KA04,20,20,20,20,80
KA15,20,20,20,20,80
KA16,20,20,20,20,80


## 3. Inspeção de um arquivo
Abrimos um arquivo para conferir a estrutura: nome, tamanho de cada canal e a duração implícita a 64 kHz. As correntes e a vibração devem durar cerca de 4 s. Os canais mecânicos (força, rotação, torque) têm 1/16 das amostras (4 kHz). A temperatura tem 1 Hz. **Por isso os canais não podem ser juntados por índice.**

In [3]:
exemplo = arquivos[0]
canais = R.ler_mat(exemplo)
print(exemplo.name)
pd.DataFrame({
    "amostras": {k: v.size for k, v in canais.items()},
    "duracao_s_a_64kHz": {k: v.size / R.FS for k, v in canais.items()},
})

N09_M07_F10_K001_1.mat


,amostras,duracao_s_a_64kHz
force,16008,0.250125
phase_current_1,256823,4.012859
phase_current_2,256823,4.012859
speed,16008,0.250125
temp_2_bearing_module,5,0.000078
torque,16008,0.250125
vibration_1,256823,4.012859


## 4. Teste em um arquivo
Rodamos `processar_arquivo` em um único arquivo. Isso confere as features geradas e mede o tempo por arquivo, que dá uma estimativa do tempo total.

Prefixos das colunas: `i1__` = corrente de fase 1, `i2__` = corrente de fase 2, `vib__` = vibração. Para cada sinal:
- **tempo**: rms, desvio, pico, pico a pico, fatores de crista/folga/impulso/forma, curtose, assimetria, integral de linha, entropia;
- **frequência**: pico e frequência do pico da FFT, energia, integral e centroide da PSD (Welch);
- **tempo-frequência**: energia relativa dos 8 nós da Wavelet Packet nível 3;
- **vibração**: amplitude do espectro de envelope em fr, BPFO (1×, 2×) e BPFI (1×, 2×), em dB acima do fundo;
- **corrente**: fundamental `f1`, bandas laterais `f1 ± BPFO` e `f1 ± BPFI` em dB relativas à fundamental, e THD.

In [4]:
t0 = time.perf_counter()
linha = R.processar_arquivo(exemplo, n_segmentos=N_SEGMENTOS)[0]
dt = time.perf_counter() - t0
print(f"{dt:.2f} s por arquivo -> estimativa sequencial para {len(arquivos)} arquivos: {dt * len(arquivos) / 60:.1f} min")
pd.Series(linha).to_frame("valor").head(40)

0.22 s por arquivo -> estimativa sequencial para 1600 arquivos: 5.9 min


,valor
arquivo,N09_M07_F10_K001_1
condicao,N09_M07_F10
rolamento,K001
medicao,1
rotacao_rpm,900
torque_nm,0.7
forca_n,1000
segmento,0
i1__rms,1.656722
i1__desvio,1.656556


## 5. Extração completa, em paralelo
`extrair_dataset` distribui os arquivos entre os núcleos. Cada processo lê um arquivo, calcula as features e devolve só um dicionário de números. Um arquivo com problema não interrompe o lote: ele vai para `erros` com a mensagem.

O CSV é salvo **sem índice** (o índice não carrega informação) e com `utf-8`.

In [5]:
t0 = time.perf_counter()
df, erros = R.extrair_dataset(arquivos, n_segmentos=N_SEGMENTOS, n_jobs=N_JOBS, verbose=0)
print(f"Extração: {time.perf_counter() - t0:.1f} s | {df.shape[0]} linhas × {df.shape[1]} colunas | {len(erros)} erros")
df.to_csv(ARQUIVO_SAIDA, index=False)
erros

488/1600 arquivos ( 30.5 %) |     38 s decorridos | ~   86 s restantes

C:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


1600/1600 arquivos (100.0 %) |    113 s decorridos | ~    0 s restantes
Extração: 112.6 s | 1600 linhas × 96 colunas | 0 erros


,arquivo,erro


In [6]:
df

,arquivo,condicao,rolamento,medicao,rotacao_rpm,torque_nm,forca_n,segmento,i1__rms,i1__desvio,...,vib__wpd_e3,vib__wpd_e4,vib__wpd_e5,vib__wpd_e6,vib__wpd_e7,vib__env_fr,vib__env_bpfo_1x,vib__env_bpfo_2x,vib__env_bpfi_1x,vib__env_bpfi_2x
0,N09_M07_F10_K001_1,N09_M07_F10,K001,1,900,0.7,1000,0,1.656722,1.656556,...,0.124036,0.123832,0.116271,0.116241,0.121244,15.744950,9.521987,13.266914,4.740935,18.169190
1,N09_M07_F10_K001_2,N09_M07_F10,K001,2,900,0.7,1000,0,1.669053,1.668909,...,0.119353,0.120253,0.118601,0.119498,0.121032,17.031448,7.924775,14.099693,6.854121,19.357992
2,N09_M07_F10_K001_3,N09_M07_F10,K001,3,900,0.7,1000,0,1.677920,1.677769,...,0.121477,0.121869,0.118166,0.115590,0.123043,17.719663,8.328271,12.196246,7.660825,17.694461
3,N09_M07_F10_K001_4,N09_M07_F10,K001,4,900,0.7,1000,0,1.682914,1.682794,...,0.120273,0.120008,0.118776,0.119466,0.120371,15.881922,8.033620,10.881797,8.081305,16.938854
4,N09_M07_F10_K001_5,N09_M07_F10,K001,5,900,0.7,1000,0,1.687410,1.687280,...,0.121667,0.121828,0.116098,0.115686,0.125628,16.825546,7.763651,11.660633,6.621630,17.063454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,N15_M07_F10_KI21_16,N15_M07_F10,KI21,16,1500,0.7,1000,0,1.850392,1.848496,...,0.013982,0.012183,0.014835,0.014215,0.012232,27.296251,17.845300,12.770627,22.731350,19.636593
1596,N15_M07_F10_KI21_17,N15_M07_F10,KI21,17,1500,0.7,1000,0,1.841524,1.840069,...,0.014426,0.012517,0.015068,0.014010,0.012398,27.809261,21.071294,10.799375,22.685219,19.313768
1597,N15_M07_F10_KI21_18,N15_M07_F10,KI21,18,1500,0.7,1000,0,1.851249,1.849409,...,0.013356,0.011448,0.014344,0.013711,0.011811,28.226172,17.159657,13.302826,22.116150,20.036476
1598,N15_M07_F10_KI21_19,N15_M07_F10,KI21,19,1500,0.7,1000,0,1.842176,1.840682,...,0.016740,0.014779,0.016589,0.016261,0.014171,28.023303,18.401649,16.471876,21.236947,19.858538


## 6. Verificação rápida
- **NaN por coluna**: um valor não finito (por exemplo, log de zero) vira NaN e depois é imputado no pipeline. Muitos NaN numa coluna indicam problema.
- **Colunas constantes**: não carregam informação e são removidas na modelagem.
- **Linhas por rolamento**: deve dar 80 × `N_SEGMENTOS`.

In [7]:
features = R.colunas_features(df)
nan = df[features].isna().sum()
print("Features:", len(features))
print("Colunas com NaN:", nan[nan > 0].to_dict())
print("Colunas constantes:", [c for c in features if df[c].nunique() <= 1])
df.groupby("rolamento").size().describe()

Features: 88
Colunas com NaN: {}
Colunas constantes: []


count    20.0
mean     80.0
std       0.0
min      80.0
25%      80.0
50%      80.0
75%      80.0
max      80.0
dtype: float64

In [10]:
df

,arquivo,condicao,rolamento,medicao,rotacao_rpm,torque_nm,forca_n,segmento,i1__rms,i1__desvio,...,vib__wpd_e3,vib__wpd_e4,vib__wpd_e5,vib__wpd_e6,vib__wpd_e7,vib__env_fr,vib__env_bpfo_1x,vib__env_bpfo_2x,vib__env_bpfi_1x,vib__env_bpfi_2x
0,N09_M07_F10_K001_1,N09_M07_F10,K001,1,900,0.7,1000,0,1.656722,1.656556,...,0.124036,0.123832,0.116271,0.116241,0.121244,15.744950,9.521987,13.266914,4.740935,18.169190
1,N09_M07_F10_K001_2,N09_M07_F10,K001,2,900,0.7,1000,0,1.669053,1.668909,...,0.119353,0.120253,0.118601,0.119498,0.121032,17.031448,7.924775,14.099693,6.854121,19.357992
2,N09_M07_F10_K001_3,N09_M07_F10,K001,3,900,0.7,1000,0,1.677920,1.677769,...,0.121477,0.121869,0.118166,0.115590,0.123043,17.719663,8.328271,12.196246,7.660825,17.694461
3,N09_M07_F10_K001_4,N09_M07_F10,K001,4,900,0.7,1000,0,1.682914,1.682794,...,0.120273,0.120008,0.118776,0.119466,0.120371,15.881922,8.033620,10.881797,8.081305,16.938854
4,N09_M07_F10_K001_5,N09_M07_F10,K001,5,900,0.7,1000,0,1.687410,1.687280,...,0.121667,0.121828,0.116098,0.115686,0.125628,16.825546,7.763651,11.660633,6.621630,17.063454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,N15_M07_F10_KI21_16,N15_M07_F10,KI21,16,1500,0.7,1000,0,1.850392,1.848496,...,0.013982,0.012183,0.014835,0.014215,0.012232,27.296251,17.845300,12.770627,22.731350,19.636593
1596,N15_M07_F10_KI21_17,N15_M07_F10,KI21,17,1500,0.7,1000,0,1.841524,1.840069,...,0.014426,0.012517,0.015068,0.014010,0.012398,27.809261,21.071294,10.799375,22.685219,19.313768
1597,N15_M07_F10_KI21_18,N15_M07_F10,KI21,18,1500,0.7,1000,0,1.851249,1.849409,...,0.013356,0.011448,0.014344,0.013711,0.011811,28.226172,17.159657,13.302826,22.116150,20.036476
1598,N15_M07_F10_KI21_19,N15_M07_F10,KI21,19,1500,0.7,1000,0,1.842176,1.840682,...,0.016740,0.014779,0.016589,0.016261,0.014171,28.023303,18.401649,16.471876,21.236947,19.858538


In [11]:
df.columns

Index(['arquivo', 'condicao', 'rolamento', 'medicao', 'rotacao_rpm',
       'torque_nm', 'forca_n', 'segmento', 'i1__rms', 'i1__desvio', 'i1__pico',
       'i1__pico_a_pico', 'i1__fator_crista', 'i1__fator_folga',
       'i1__fator_impulso', 'i1__fator_forma', 'i1__curtose', 'i1__assimetria',
       'i1__integral_linha', 'i1__entropia', 'i1__fft_pico',
       'i1__fft_freq_pico', 'i1__fft_energia', 'i1__psd_integral',
       'i1__centroide_espectral', 'i1__wpd_e0', 'i1__wpd_e1', 'i1__wpd_e2',
       'i1__wpd_e3', 'i1__wpd_e4', 'i1__wpd_e5', 'i1__wpd_e6', 'i1__wpd_e7',
       'i1__f1_hz', 'i1__lat_bpfo_db', 'i1__lat_bpfi_db', 'i1__thd', 'i2__rms',
       'i2__desvio', 'i2__pico', 'i2__pico_a_pico', 'i2__fator_crista',
       'i2__fator_folga', 'i2__fator_impulso', 'i2__fator_forma',
       'i2__curtose', 'i2__assimetria', 'i2__integral_linha', 'i2__entropia',
       'i2__fft_pico', 'i2__fft_freq_pico', 'i2__fft_energia',
       'i2__psd_integral', 'i2__centroide_espectral', 'i2__wpd_e0'